# 04 - Reproduce hybrid count and pagination behavior

**Hypothesis:** a hybrid count can describe more matches than its pageable candidate set. This experiment measures the symptom; it does not assume the customer's 257-page boundary or prove an SDK defect.

First complete [initial infrastructure readiness](03_test.ipynb) on the current empty service. This notebook creates one synthetic index, uploads 6,158 records, and makes several thousand sequential reads. No model is needed. Never run preparation during either client's measurements. Do not rerun the empty-service readiness after ingestion or delete data just to satisfy it.

Use the demo Python kernel and `notebooks/` working directory. Both clients read the same generated environment and immutable fixture. A new complete experiment needs a new run directory; existing case evidence is never overwritten.

## Load experiment prerequisites

Check both toolchains and the exported setup-user context before mutations. This aligns the shared Azure CLI default subscription to the export, without changing login identity.

In [ ]:
import json
import subprocess
import sys
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

demo_root = Path('..').resolve()
sys.path.insert(0, str(demo_root))
from app.fixture import API, COUNT, INDEX, generate, load_environment, prepare
from app.reproduce import run
from app.compare import compare
from azure.identity import AzureCliCredential
from azure.search.documents import SearchClient

if sys.version_info < (3, 11) or version('azure-search-documents') != '12.0.0' or version('azure-identity') != '1.25.3':
    raise RuntimeError('Use Python 3.11+ and the pinned setup requirements.')
sdk_check = subprocess.run(['dotnet', '--list-sdks'], capture_output=True, text=True)
if sdk_check.returncode or not any(line.startswith('10.') for line in sdk_check.stdout.splitlines()):
    raise RuntimeError(f'.NET 10 SDK required, not just a runtime.\n{sdk_check.stdout}\n{sdk_check.stderr}')
env_file = demo_root / 'env' / '.env'
environment = load_environment(env_file)
generated = demo_root / 'outputs' / 'generated'
readiness = json.loads((generated / 'readiness.json').read_text(encoding='utf-8'))
if readiness['status'] != 'ready' or readiness['environment'] != environment or readiness['counters'] != {'indexes': 0, 'documents': 0}:
    raise ValueError('Complete current initial infrastructure readiness before preparing the index.')
project = demo_root / 'app' / 'dotnet' / 'HybridPagination.csproj'
restore = subprocess.run(['dotnet', 'restore', str(project), '--locked-mode'], capture_output=True, text=True)
print(restore.stdout)
if restore.returncode:
    raise RuntimeError(f'.NET restore failed ({restore.returncode}):\n{restore.stderr}')
build = subprocess.run(['dotnet', 'build', str(project), '--no-restore'], capture_output=True, text=True)
print(build.stdout)
if build.returncode:
    raise RuntimeError(f'.NET build failed ({build.returncode}):\n{build.stderr}')
print(f'Python {sys.version.split()[0]}, Search {version("azure-search-documents")}, API {API}')
print(sdk_check.stdout)


## Generate the fixed synthetic fixture

Content order: topic, ID, repeated `maths`, repeated `lesson`. SHA-256 bytes determine every value. Compact UTF-8 JSONL with LF endings and a declared-schema hash gives both clients identical inputs.

In [ ]:
reproduction = generated / 'reproduction'
manifest_path = generate(reproduction / 'fixture')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:8]
run_dir = reproduction / 'runs' / run_id
print(json.dumps(manifest, indent=2))
print(f'Evidence directory: {run_dir}')


## Prepare the shared Search index

Run once before measurements. Uses the existing keyless service, exhaustive cosine search, and deterministic IDs. Every indexing result is checked. No other index is cleared and no infrastructure is deployed.

In [ ]:
print(json.dumps(prepare(manifest_path, env_file), indent=2))


## Confirm the corpus is indexed

Require exactly 6,158 indexed records. If visibility has not settled, wait and rerun this check only; do not start a partial experiment. From this point until comparison, make no index or document writes.

In [ ]:
with AzureCliCredential(subscription=environment['AZURE_SUBSCRIPTION_ID']) as credential:
    with SearchClient(environment['AZURE_SEARCH_ENDPOINT'], INDEX, credential, api_version=API) as client:
        indexed_count = client.get_document_count()
if indexed_count != COUNT:
    raise RuntimeError(f'Indexed {indexed_count}; require {COUNT}. Wait and rerun this check.')
print(f'Immutable measurement input ready: {indexed_count} records.')


## Run the Python keyword control

The baseline must reach all 6,158 IDs, with 1,232 populated pages, no duplicates, and three final-page rows. Failure blocks causal attribution, not evidence preservation.

In [ ]:
python_keyword_exit = run('keyword', manifest_path, env_file, run_dir)
if python_keyword_exit:
    raise RuntimeError(f'Python keyword incomplete; inspect {run_dir / "python-keyword.json"}. Continue other independent cases only for diagnosis.')


## Run the Python vector control

No lexical search or wildcard. If the service omits the initial count, scan all ten candidate pages plus boundaries; count-based fields stay null and successful completion is `not-applicable`, not proof of no discrepancy.

In [ ]:
python_vector_exit = run('vector', manifest_path, env_file, run_dir)
if python_vector_exit:
    raise RuntimeError(f'Python vector incomplete; inspect {run_dir / "python-vector.json"}.')


## Run the Python hybrid case

Only the offset changes within this case. Every advertised page is measured, including empty pages. The runner retains JSON/JSONL evidence and repeats deduplicated boundaries exactly once. Short responses with continuation are incomplete, never exhausted-page evidence.

In [ ]:
python_hybrid_exit = run('hybrid', manifest_path, env_file, run_dir)
if python_hybrid_exit:
    raise RuntimeError(f'Python hybrid incomplete; inspect {run_dir / "python-hybrid.json"}.')


## Run the .NET keyword control

This is an independent direct .NET Search SDK client, not a wrapper around Python results. It reads the same manifest, corpus and generated environment.

In [ ]:
dotnet_arguments = ['dotnet', 'run', '--no-build', '--project', str(project), '--', '--manifest', str(manifest_path), '--env-file', str(env_file), '--run-dir', str(run_dir)]
result = subprocess.run([*dotnet_arguments, '--case', 'keyword'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError(f'.NET keyword failed ({result.returncode}); inspect its retained evidence.')


## Run the .NET vector control


In [ ]:
result = subprocess.run([*dotnet_arguments, '--case', 'vector'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError(f'.NET vector failed ({result.returncode}); inspect its retained evidence.')


## Run the .NET hybrid case


In [ ]:
result = subprocess.run([*dotnet_arguments, '--case', 'hybrid'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError(f'.NET hybrid failed ({result.returncode}); inspect its retained evidence.')


## Compare observations and explain the finding

Local recomputation only: no replacement searches. Missing cases, unstable boundaries, failures, and request mismatches remain gaps. Reproduction requires both a positive count-minus-unique gap and an empty advertised page. A valid `not-reproduced` result also completes the experiment.

In [ ]:
from IPython.display import FileLink, Markdown, display
comparison = compare(run_dir)
display(Markdown((run_dir / 'comparison.md').read_text(encoding='utf-8')))
for evidence in sorted(run_dir.glob('*')):
    display(FileLink(str(Path('..') / evidence.relative_to(demo_root))))
if comparison['status'] != 'complete':
    raise RuntimeError('Comparison is incomplete. Inspect the retained gaps; no customer-cause conclusion is justified.')


## Finish or run again

Keep evidence before any cleanup. For another full experiment, run **Generate the fixed synthetic fixture** to create a new run directory, skip index preparation if the index is unchanged, and rerun the measurements. Counts are not snapshot isolation. For intentional deletion, review the canonical [destroy notebook](X_destroy.ipynb) separately; do not run it unattended. Paid billing continues until cleanup.